In [1]:
import pandas as pd
import numpy as np
from lightgbm import LGBMRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import LabelEncoder

In [2]:
train = pd.read_csv('dataTrain_Spotify.csv')
test = pd.read_csv('dataTest_Spotify.csv')

test_ids = test.index

In [3]:
# limpiar columnas problemáticas
train = train.drop(columns=['Unnamed: 0'], errors='ignore')
test = test.drop(columns=['Unnamed: 0','Unnamed0'], errors='ignore')

In [ ]:
# convertir texto a string limpio para usar light
for col in ['track_genre']:
    train[col] = train[col].astype(str)
    test[col] = test[col].astype(str)

In [5]:
# encoding de genero
le = LabelEncoder()
train['track_genre'] = le.fit_transform(train['track_genre'])
test['track_genre'] = test['track_genre'].map(dict(zip(le.classes_, le.transform(le.classes_))))
test['track_genre'] = test['track_genre'].fillna(-1)

In [6]:
# variables nuevas
train['duration_min'] = train['duration_ms'] / 60000
test['duration_min'] = test['duration_ms'] / 60000

train['energy_valence'] = train['energy'] * train['valence']
test['energy_valence'] = test['energy'] * test['valence']

In [7]:
# convertir booleanos
train['explicit'] = train['explicit'].astype(int)
test['explicit'] = test['explicit'].astype(int)

In [8]:
# eliminar columnas de texto
drop_cols = ['track_id','track_name','album_name','artists']
train = train.drop(columns=drop_cols, errors='ignore')
test = test.drop(columns=drop_cols, errors='ignore')

In [9]:
# separar variables
X = train.drop(columns=['popularity'])
y = train['popularity']

test = test[X.columns]

In [10]:
# renombrar columnas para evitar error LightGBM
X.columns = [f'col_{i}' for i in range(X.shape[1])]
test.columns = X.columns

In [11]:
# KFold
kf = KFold(n_splits=5, shuffle=True, random_state=42)

preds_test = np.zeros(len(test))
oof = np.zeros(len(X))

In [12]:
# entrenamiento
for train_idx, val_idx in kf.split(X):

    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model = LGBMRegressor(n_estimators=1000, learning_rate=0.03, random_state=42)
    model.fit(X_train, y_train)

    preds_val = model.predict(X_val)
    oof[val_idx] = preds_val

    preds_test += model.predict(test) / 5

    rmse = mean_squared_error(y_val, preds_val) ** 0.5
    print('RMSE fold:', rmse)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002191 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3194
[LightGBM] [Info] Number of data points in the train set: 63840, number of used features: 17
[LightGBM] [Info] Start training from score 33.324185
RMSE fold: 17.336673096149415
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002107 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3195
[LightGBM] [Info] Number of data points in the train set: 63840, number of used features: 17
[LightGBM] [Info] Start training from score 33.261466
RMSE fold: 17.67235129242571
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001951 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3195
[LightGBM] [Info] Number of data points in the train set: 6

In [ ]:
preds_test = np.clip(preds_test, 0, 100)

submission = pd.DataFrame({
    'ID': test_ids,
    'Popularity': preds_test
})

submission.to_csv('submission_lgb_final.csv', index=False)


Archivo generado correctamente
